# orca-colab bootstrap

Connects this Google Colab runtime to your Tailscale tailnet and starts `sshd` so your local Orca workspace and coding agent can use this GPU as a disposable compute backend.

1. **Pick a GPU runtime first:** `Runtime → Change runtime type → Hardware accelerator` (e.g. **T4 GPU**).
2. **Fill the form below** (or save a personal copy in Google Drive with your values prefilled; or paste `<authkey> <pubkey>` from your clipboard into `PASTE`) and press **Run**.

> **Security note:** Use a reusable, ephemeral, tagged (`tag:orca-colab`) Tailscale auth key and never share or commit a notebook copy that contains your auth key.

In [ ]:
#@title Connect this runtime to your tailnet for orca-colab
TAILSCALE_AUTH_KEY = ""     #@param {type:"string"}
SSH_PUBLIC_KEY = ""         #@param {type:"string"}
TAILSCALE_LOGIN_SERVER = "" #@param {type:"string"}
HOSTNAME_SUFFIX = ""        #@param {type:"string"}
PASTE = ""                  #@param {type:"string"}

import os
import pathlib
import secrets
import shutil
import subprocess
import threading
import time

auth_key = TAILSCALE_AUTH_KEY.strip()
ssh_pub_key = SSH_PUBLIC_KEY.strip()
login_server = TAILSCALE_LOGIN_SERVER.strip()
suffix = HOSTNAME_SUFFIX.strip().lower()

if PASTE.strip():
    raw_paste = PASTE.strip()
    key_idx = -1
    for marker in ("ssh-ed25519 ", "ssh-rsa "):
        idx = raw_paste.find(marker)
        if idx > 0:
            key_idx = idx
            break
    if key_idx <= 0:
        raise ValueError("PASTE must contain '<authkey> [<login_server>] <pubkey>'.")
    prefix_tokens = raw_paste[:key_idx].strip().split()
    ssh_pub_key = raw_paste[key_idx:].strip()
    if len(prefix_tokens) == 1:
        auth_key = prefix_tokens[0]
    elif len(prefix_tokens) == 2:
        auth_key, login_server = prefix_tokens[0], prefix_tokens[1]
    else:
        raise ValueError("PASTE must contain '<authkey> [<login_server>] <pubkey>'.")

if not auth_key or (not login_server and not auth_key.startswith("tskey-")) or (login_server and len(auth_key) < 16):
    raise ValueError("Invalid TAILSCALE_AUTH_KEY: must start with 'tskey-' (or be a >=16-char Headscale pre-auth key with TAILSCALE_LOGIN_SERVER set).")

if not (ssh_pub_key.startswith("ssh-ed25519 ") or ssh_pub_key.startswith("ssh-rsa ")):
    raise ValueError("Invalid SSH_PUBLIC_KEY: must start with 'ssh-ed25519 ' or 'ssh-rsa '.")

if not suffix:
    suffix = secrets.token_hex(3)
hostname = suffix if suffix.startswith("orca-colab-") else f"orca-colab-{suffix}"

print("[orca-colab] Installing openssh-server and rsync...")
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(
    ["apt-get", "install", "-y", "-qq", "openssh-server", "rsync"],
    check=True,
    env={**os.environ, "DEBIAN_FRONTEND": "noninteractive"},
)

ssh_dir = pathlib.Path("/root/.ssh")
ssh_dir.mkdir(mode=0o700, parents=True, exist_ok=True)
os.chmod(ssh_dir, 0o700)
auth_keys_path = ssh_dir / "authorized_keys"
auth_keys_path.write_text(ssh_pub_key + "\n", encoding="utf-8")
os.chmod(auth_keys_path, 0o600)

pathlib.Path("/run/sshd").mkdir(mode=0o755, parents=True, exist_ok=True)
sshd_cfg = pathlib.Path("/etc/ssh/sshd_config.d/orca-colab.conf")
sshd_cfg.parent.mkdir(parents=True, exist_ok=True)
sshd_cfg.write_text(
    "Port 22\n"
    "Port 2222\n"
    "ListenAddress 0.0.0.0\n"
    "MaxStartups 10:30:100\n"
    "PermitRootLogin prohibit-password\n"
    "PasswordAuthentication no\n"
    "PubkeyAuthentication yes\n",
    encoding="utf-8",
)
main_sshd_cfg = pathlib.Path("/etc/ssh/sshd_config")
if main_sshd_cfg.exists():
    existing_cfg = main_sshd_cfg.read_text(encoding="utf-8")
    if "Include /etc/ssh/sshd_config.d/*.conf" not in existing_cfg:
        main_sshd_cfg.write_text(
            "Include /etc/ssh/sshd_config.d/*.conf\n" + existing_cfg,
            encoding="utf-8",
        )
subprocess.run(["service", "ssh", "restart"], check=True)

if not shutil.which("tailscale"):
    print("[orca-colab] Installing Tailscale...")
    subprocess.run("curl -fsSL https://tailscale.com/install.sh | sh", shell=True, check=True)

subprocess.run(["pkill", "-f", "tailscaled"], check=False)
tailscaled_log = open("/root/tailscaled.log", "ab")
subprocess.Popen(
    [
        "tailscaled",
        "--tun=userspace-networking",
        "--socks5-server=localhost:1055",
        "--state=/root/tailscaled.state",
    ],
    stdout=tailscaled_log,
    stderr=subprocess.STDOUT,
)
time.sleep(2)

ts_up_cmd = [
    "tailscale",
    "up",
    f"--authkey={auth_key}",
    f"--hostname={hostname}",
    "--accept-dns=false",
    "--accept-routes=false",
]
if login_server:
    ts_up_cmd.append(f"--login-server={login_server}")
subprocess.run(ts_up_cmd, check=True)

state_dir = pathlib.Path("/root/.orca-colab")
state_dir.mkdir(mode=0o700, parents=True, exist_ok=True)
unassign_sentinel = state_dir / "unassign"
if unassign_sentinel.exists():
    unassign_sentinel.unlink()
(state_dir / "ready").write_text(f"{hostname}\n", encoding="utf-8")

def _watch_unassign():
    while True:
        if unassign_sentinel.exists():
            subprocess.run(["service", "ssh", "stop"], check=False)
            subprocess.run(["pkill", "-f", "sshd"], check=False)
            subprocess.run(["tailscale", "down"], check=False)
            try:
                from google.colab import runtime
                runtime.unassign()
            except Exception as exc:
                print(f"[orca-colab] runtime.unassign() failed: {exc}")
            break
        time.sleep(1)

threading.Thread(target=_watch_unassign, name="orca-colab-unassign", daemon=True).start()

gpu_proc = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
if gpu_proc.returncode == 0 and gpu_proc.stdout.strip():
    print(gpu_proc.stdout.strip())
else:
    print("[orca-colab] No NVIDIA GPU detected (CPU runtime — use Runtime -> Change runtime type for a GPU).")

ip_proc = subprocess.run(["tailscale", "ip", "-4"], capture_output=True, text=True, check=True)
print(f"Hostname: {hostname}")
print(f"Tailscale IPv4: {ip_proc.stdout.strip()}")
print("Connected. Keep this tab open.")

In [ ]:
# Optional keepalive loop (not auto-run). Touches a file every 5 minutes.
# Note: this keeps the kernel busy, but does NOT defeat Colab's browser-tab idle check — keep the tab open.
import pathlib
import time

keepalive_path = pathlib.Path("/root/.orca-colab/keepalive")
keepalive_path.parent.mkdir(mode=0o700, parents=True, exist_ok=True)
try:
    while True:
        keepalive_path.write_text(str(int(time.time())) + "\n", encoding="utf-8")
        time.sleep(300)
except KeyboardInterrupt:
    print("Keepalive stopped.")